# Bahraini Food YOLO — Clean Google Colab Notebook
Single Roboflow dataset → analyze → train → evaluate → camera test

## 1. Install

In [ ]:
!pip install -q ultralytics roboflow pyyaml

## 2. Imports

In [ ]:
import os, glob, yaml, shutil
from collections import Counter, defaultdict
from getpass import getpass
import matplotlib.pyplot as plt
from roboflow import Roboflow
from ultralytics import YOLO


## 3. Roboflow login

In [ ]:
API_KEY = getpass("Enter Roboflow API key: ")
rf = Roboflow(api_key=API_KEY)


## 4. Download dataset

In [ ]:
project = rf.workspace("mohammad-hamza-sadiq").project("bahrainifood")
version = project.version(1)
dataset = version.download("yolo26")

DATASET_DIR = dataset.location
yaml_path = os.path.join(DATASET_DIR, "data.yaml")

print(DATASET_DIR)


## 5. Check classes

In [ ]:
with open(yaml_path, "r") as f:
    data = yaml.safe_load(f)

class_names = data["names"]
print(class_names)


## 6. Dataset counts

In [ ]:
for split in ["train", "valid", "test"]:
    print(split, ":", len(glob.glob(f"{DATASET_DIR}/{split}/images/*")))


## 7. Object counts

In [ ]:
def names_list(names):
    if isinstance(names, dict):
        return [names.get(i, names.get(str(i))) for i in range(len(names))]
    return list(names)

class_list = names_list(class_names)
class_counts = Counter()
images_per_class = defaultdict(set)
total_images = 0

for split in ["train", "valid", "test"]:
    total_images += len(glob.glob(f"{DATASET_DIR}/{split}/images/*"))

    for label_file in glob.glob(f"{DATASET_DIR}/{split}/labels/*.txt"):
        seen = set()

        for line in open(label_file):
            parts = line.split()
            if not parts:
                continue
            cid = int(parts[0])
            class_counts[cid] += 1
            seen.add(cid)

        for cid in seen:
            images_per_class[cid].add(label_file)

total_objects = sum(class_counts.values())

print("Total images:", total_images)
print("Total objects:", total_objects)
print("Average objects/image:", round(total_objects / total_images, 2))

for i, name in enumerate(class_list):
    print(f"{name}: {class_counts[i]} objects | {len(images_per_class[i])} images")


## 8. Class chart

In [ ]:
counts = [class_counts[i] for i in range(len(class_list))]

plt.figure(figsize=(12, 6))
plt.bar(class_list, counts)
plt.xlabel("Food Class")
plt.ylabel("Annotated Objects")
plt.title("Annotated Objects per Class")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 9. Baseline

In [ ]:
model = YOLO("yolo26n.pt")

test_images = glob.glob(f"{DATASET_DIR}/test/images/*")
if not test_images:
    raise ValueError("No test images found.")

baseline = model(test_images[0])
baseline[0].show()


## 10. Train with early stopping

In [ ]:
shutil.rmtree("/content/runs/bahraini_food/v1", ignore_errors=True)

model = YOLO("yolo26n.pt")

model.train(
    data=yaml_path,
    epochs=100,
    imgsz=640,
    batch=8,
    patience=15,
    project="/content/runs/bahraini_food",
    name="v1"
)


## 11. Load best model

In [ ]:
best_path = "/content/runs/bahraini_food/v1/weights/best.pt"
v1_model = YOLO(best_path)
print(v1_model.names)


## 12. Evaluate

In [ ]:
metrics = v1_model.val(data=yaml_path, split="test")

print("Precision:", round(metrics.box.mp, 4))
print("Recall:", round(metrics.box.mr, 4))
print("mAP50:", round(metrics.box.map50, 4))
print("mAP50-95:", round(metrics.box.map, 4))


## 13. Per-class mAP50-95

In [ ]:
for name, score in zip(v1_model.names.values(), metrics.box.maps):
    print(name, ":", round(float(score), 4))


## 14. Test images

In [ ]:
results = v1_model(test_images[:10], conf=0.25)

for result in results:
    result.show()


## 15. Camera setup

In [ ]:
from google.colab.output import eval_js
from base64 import b64decode
from IPython.display import Javascript, display

def take_photo(filename="camera_test.jpg"):
    js = Javascript("""
    async function takePhoto() {
      const div = document.createElement('div');
      const video = document.createElement('video');
      const button = document.createElement('button');

      button.textContent = 'Take Photo';
      div.appendChild(video);
      div.appendChild(document.createElement('br'));
      div.appendChild(button);
      document.body.appendChild(div);

      const stream = await navigator.mediaDevices.getUserMedia({video: true});
      video.srcObject = stream;
      await video.play();

      await new Promise(resolve => button.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);

      stream.getTracks().forEach(track => track.stop());
      div.remove();

      return canvas.toDataURL('image/jpeg');
    }
    """)

    display(js)
    data = eval_js("takePhoto()")
    binary = b64decode(data.split(",")[1])

    with open(filename, "wb") as f:
        f.write(binary)

    return filename


## 16. Camera detection

In [ ]:
camera_image = take_photo()

result = v1_model(camera_image, conf=0.25)
result[0].show()

if len(result[0].boxes) == 0:
    print("No detections.")
else:
    for box in result[0].boxes:
        cid = int(box.cls[0])
        conf = float(box.conf[0])
        print(v1_model.names[cid], f"{conf * 100:.1f}%")


For debugging only, if the camera shows no detection, try `conf=0.10`.